# 05. 장애인콜택시 의료 목적지 × 의료기관 분포 분석 — 목적 정의 및 데이터 준비

이 노트북은 STEP1(목적 정의)과 STEP2(데이터 수집 및 준비)를 다룬다.
작업계획: `notebooks_docs_lye/plan_medical_destination_analysis.md` 참고.

## STEP1. 목적 정의

**분석 배경**
장애인콜택시는 장애인의 이동권 보장을 위한 교통수단이며, 병원 등 의료기관을 목적지로 이용하는 사례가 중요한 이용 유형 중 하나라고 가정한다. 다만 OD 데이터만으로는 실제 방문한 병원명·주소·좌표를 알 수 없고, 목적지 정보는 `목적구/목적동` 수준으로만 확인 가능하다.

**분석 목적**
개별 탑승건과 특정 병원을 매칭하지 않고, 「장애인콜택시 목적지 지역(구·동)」과 「해당 지역의 의료기관 분포」를 결합하여 목적지 지역별 의료환경과 콜택시 이용량 간의 탐색적 관계를 확인한다.

**핵심 연구 질문**
1. 목적지 건수가 많은 구·동은 어디인가?
2. 목적지 건수 높은 지역엔 어떤 의료기관 종별이 많은가?
3. 목적지 건수 상/하위 지역의 의료기관 구성 차이는?
4. 의원 중심 지역 vs 병원급 중심 지역의 목적지 패턴 차이는?
5. 상급종합/종합병원 존재 지역의 목적지 건수는 상대적으로 높은가?
6. 요양병원 많은 지역의 목적지 건수 특성은?
7. 의료기관 총수와 목적지 건수 간 통계적으로 유의미한 관계가 있는가?
8. 의료기관 '수'뿐 아니라 '구성'이 목적지 특성 설명에 도움이 되는가?
9. 의료기관 분포 × 목적지 분포 결합 시 특징적 지역 유형을 구분할 수 있는가?

**분석 대상 / 데이터 범위**
- 장애인콜택시: `data/processed/서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv` (원본, **수정 금지** — 이 노트북에서는 읽기 전용으로만 사용)
- 의료기관: `data/raw/hospital_info_seoul.csv` (HIRA 병원정보서비스, 서울시 P1+P2 우선순위 14,856건)
  - 좌표(XPos/YPos)를 카카오 로컬 API로 역지오코딩해 `행정동` 컬럼을 추가한 `data/processed/hospital_info_seoul_행정동_파생컬럼추가.csv`를 실제 분석에 사용 (raw 원본은 미수정)
  - ⚠️ **서울시 전체 의료기관이 아니라 P1(핵심 의료 목적지: 종합병원, 요양병원, 의원 등
 )+P2(지역사회·특수 의료 목적지: 보건소, 한의원 등) 우선순위군만 선별 수집**한 데이터다. 선정 근거와 출처는 2-5절 참고.

**분석 단위**: 「목적지 구·동」을 주 분석단위로 채택 (아래 2-6절에서 매칭 검증, 최종 매칭률 99.72%). 「목적지 구」 단위 집계도 보조 산출물로 함께 생성한다.

**분석의 한계**
- 개별 탑승건이 특정 병원을 방문했다고 단정하지 않는다
- 의료기관 진료과목(재활의학과 등)은 현재 데이터로 분석 불가
- 상관관계가 확인되어도 인과관계로 해석하지 않는다

**검증할 가설**
의료기관 수·구성이 다른 지역 간에 장애인콜택시 목적지 건수 패턴에 차이가 있을 것이다 (탐색적 가설, 이번 노트북에서는 검증하지 않고 STEP4에서 다룸).


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 60)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = PROCESSED_DIR  # 결합 산출물 저장 위치 (data/processed)

boarding_path = PROCESSED_DIR / "서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv"
hira_raw_path = RAW_DIR / "hospital_info_seoul.csv"  # 참고용(사용 안 함, 미수정 원본)
hira_path = PROCESSED_DIR / "hospital_info_seoul_행정동_파생컬럼추가.csv"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"boarding_path exists: {boarding_path.exists()}")
print(f"hira_path exists: {hira_path.exists()}")


PROJECT_ROOT: C:\Users\young\my_project\260901_call_taxi_DA
boarding_path exists: True
hira_path exists: True


## STEP2. 데이터 수집 및 준비

### 2-1. 콜택시 데이터 확인 (의료기관 데이터 확인은 2-5절에서 진행)

두 데이터를 각각 읽어와 shape, 컬럼, dtype을 확인한다. 콜택시_탑승내역 정제 데이터는 원본 파일이므로 이 노트북에서는 **읽기만 하고 절대 덮어쓰지 않는다.**


In [2]:
df_boarding = pd.read_csv(boarding_path, encoding="utf-8-sig", low_memory=False)

print(f"df_boarding shape: {df_boarding.shape}")
df_boarding.info()


df_boarding shape: (1722490, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1722490 entries, 0 to 1722489
Data columns (total 15 columns):
 #   Column  Dtype  
---  ------  -----  
 0   접수일시    object 
 1   예정일시    object 
 2   배차일시    object 
 3   승차일시    object 
 4   하차일시    object 
 5   취소일시    object 
 6   출발구     object 
 7   출발동     object 
 8   목적구     object 
 9   목적동     object 
 10  이용목적    object 
 11  요금      float64
 12  승차거리    int64  
 13  차량구분    object 
 14  장애유형    object 
dtypes: float64(1), int64(1), object(13)
memory usage: 197.1+ MB


In [3]:
# clCd는 '01'처럼 앞자리 0이 있는 코드라 정수로 읽으면 손실되므로 문자열로 강제 지정.
# 원본 CSV 자체에도 '01'이 '1'로 0패딩이 빠진 채 저장되어 있어 zfill(2)로 복원한다.
#
# hospital_info_seoul.csv(raw)에는 법정동(emdongNm)만 있어 콜택시 목적동(행정동)과
# 표기 체계가 달라 직접 매칭하면 매칭률이 18.7%에 불과했다(2-6절 참고). 이를 해결하기 위해
# notebooks_docs_lye/add_admin_dong.py 스크립트로 병원 좌표(XPos, YPos) 8,414개(고유값)를
# 카카오 로컬 API(coord2regioncode, region_type=H)로 역지오코딩하여 '행정동' 컬럼을 추가한
# 가공 데이터를 사용한다 (조회 실패 0건, raw 원본은 수정하지 않음).
df_hira = pd.read_csv(hira_path, encoding="utf-8-sig", dtype={"clCd": str})
df_hira["clCd"] = df_hira["clCd"].str.zfill(2)

print(f"df_hira shape: {df_hira.shape}")
df_hira.info()


df_hira shape: (14856, 34)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14856 entries, 0 to 14855
Data columns (total 34 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   addr            14856 non-null  object 
 1   clCd            14856 non-null  object 
 2   clCdNm          14856 non-null  object 
 3   cmdcGdrCnt      14856 non-null  int64  
 4   cmdcIntnCnt     14856 non-null  int64  
 5   cmdcResdntCnt   14856 non-null  int64  
 6   cmdcSdrCnt      14856 non-null  int64  
 7   detyGdrCnt      14856 non-null  int64  
 8   detyIntnCnt     14856 non-null  int64  
 9   detyResdntCnt   14856 non-null  int64  
 10  detySdrCnt      14856 non-null  int64  
 11  drTotCnt        14856 non-null  int64  
 12  emdongNm        14850 non-null  object 
 13  estbDd          14856 non-null  int64  
 14  hospUrl         2501 non-null   object 
 15  mdeptGdrCnt     14856 non-null  int64  
 16  mdeptIntnCnt    14856 non-null  int64  
 17  mdep

### 2-2. 실제 이용건 정의

기존 `02_od_flow_same_vs_diff_gu.ipynb`, `03_same_gu_movement_analysis.ipynb`, `04_inter_gu_movement_analysis.ipynb`와 동일한 기준을 채택한다.

**"실제 이용건" = 탑승완료 (승차일시 존재 AND 하차일시 존재 AND 취소일시 결측)**

이유: 배차 실패·취소 건은 실제 의료기관 방문으로 이어지지 않았을 가능성이 높고, 기존 이동 흐름 분석 노트북들과 동일 기준을 유지해야 이번 분석 결과를 기존 분석과 비교할 수 있다.


In [4]:
missing_ride = df_boarding["승차일시"].isna().sum()
missing_drop = df_boarding["하차일시"].isna().sum()
has_cancel = df_boarding["취소일시"].notna().sum()

print(f"전체 행 수: {len(df_boarding):,}")
print(f"승차일시 결측: {missing_ride:,}건")
print(f"하차일시 결측: {missing_drop:,}건")
print(f"취소일시 존재(취소건): {has_cancel:,}건")

completed = df_boarding[
    df_boarding["승차일시"].notna()
    & df_boarding["하차일시"].notna()
    & df_boarding["취소일시"].isna()
].copy()

print(f"\n탑승완료 건수: {len(completed):,} / 전체 {len(df_boarding):,} ({len(completed) / len(df_boarding) * 100:.1f}%)")


전체 행 수: 1,722,490
승차일시 결측: 245,361건
하차일시 결측: 245,423건
취소일시 존재(취소건): 245,297건



탑승완료 건수: 1,477,067 / 전체 1,722,490 (85.8%)


### 2-3. 목적지 지역 정제

`목적구` 결측치와 서울 25개 자치구 외 지역 여부를 확인한다. 의료기관 데이터가 서울시 전용이므로, 이후 결합 분석은 목적구가 서울 25개 자치구인 건만 대상으로 한다.


In [5]:
SEOUL_25 = [
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
]

missing_dest_gu = completed["목적구"].isna().sum()
non_seoul_dest = completed[
    completed["목적구"].notna() & ~completed["목적구"].isin(SEOUL_25)
]

print(f"탑승완료 건수: {len(completed):,}")
print(f"목적구 결측: {missing_dest_gu:,}건")
print(f"서울 25개 자치구 외 목적구: {len(non_seoul_dest):,}건 ({len(non_seoul_dest) / len(completed) * 100:.2f}%)")
print(f"서울 외 목적구 상위 10개:")
print(non_seoul_dest["목적구"].value_counts().head(10))

before_rows = len(completed)
completed_seoul_dest = completed[completed["목적구"].isin(SEOUL_25)].copy()
after_rows = len(completed_seoul_dest)

print(f"\n[제외 기준] 목적구가 서울 25개 자치구가 아니거나 결측인 행 제외")
print(f"제외 전: {before_rows:,} / 제외 후: {after_rows:,} / 제외됨: {before_rows - after_rows:,}건")


탑승완료 건수: 1,477,067
목적구 결측: 0건
서울 25개 자치구 외 목적구: 107,784건 (7.30%)
서울 외 목적구 상위 10개:
목적구
하남시        12493
고양시덕양구      9957
남양주시        9895
광명시         8130
의정부시        8060
성남시분당구      7439
김포시         6863
고양시일산동구     5565
부천시원미구      4584
구리시         4457
Name: count, dtype: int64



[제외 기준] 목적구가 서울 25개 자치구가 아니거나 결측인 행 제외
제외 전: 1,477,067 / 제외 후: 1,369,283 / 제외됨: 107,784건


### 2-4. 목적동(행정동) 결측치 확인

동 단위 결합을 시도하기 전에, 콜택시 데이터의 `목적동`(행정동) 자체에 결측치가 있는지 먼저 확인한다. (행정동 vs 법정동 표기 불일치와는 별개로, 값 자체가 비어있는 행이 있는지 확인하는 단계)

In [6]:
missing_dest_dong_all = completed["목적동"].isna().sum()
missing_dest_dong_seoul = completed_seoul_dest["목적동"].isna().sum()

print(f"탑승완료 전체 중 목적동 결측: {missing_dest_dong_all:,}건 / {len(completed):,}건")
print(f"서울 25개구 목적지로 필터링 후 목적동 결측: {missing_dest_dong_seoul:,}건 / {len(completed_seoul_dest):,}건")


탑승완료 전체 중 목적동 결측: 0건 / 1,477,067건
서울 25개구 목적지로 필터링 후 목적동 결측: 0건 / 1,369,283건


### 2-5. 의료기관 clCd 코드 검증

수집 시 합의한 clCd 코드(01/11/21/28/29/31/41/51/61/71/72/93) 외의 값이 있는지 확인하고, 주요 컬럼(`emdongNm`, `hospUrl`, `telno`, `sgguCdNm`)의 결측치를 확인한다.


In [7]:
EXPECTED_CL_CODES = {"01", "11", "21", "28", "29", "31", "41", "51", "61", "71", "72", "93"}
actual_codes = set(df_hira["clCd"].unique())
unexpected_codes = actual_codes - EXPECTED_CL_CODES

print("clCd 분포:")
print(df_hira["clCd"].value_counts())
print(f"\n예상 외 clCd 값: {unexpected_codes if unexpected_codes else '없음'}")

print("\n주요 컬럼 결측치:")
for col in ["sgguCdNm", "emdongNm", "hospUrl", "telno", "clCdNm"]:
    print(f"  {col}: {df_hira[col].isna().sum():,}건")


clCd 분포:
clCd
31    10696
93     3719
21      232
28      101
11       45
71       25
01       14
29       13
72       11
Name: count, dtype: int64

예상 외 clCd 값: 없음

주요 컬럼 결측치:
  sgguCdNm: 0건
  emdongNm: 6건
  hospUrl: 12,355건
  telno: 240건
  clCdNm: 0건


### 의료기관 유형별 수집 범위(P1+P2) 선정 근거

위 clCd 분포에서 보듯, 이번에 수집한 9개 종별(01/11/21/28/31/71/72/93/29)은 임의로 고른 게 아니라 아래 근거에 따라 P1(핵심)/P2(보조)로 나눠 선정한 것이다. 41(치과병원)·51(치과의원)·61(조산원)은 이 근거에 따라 처음부터 수집 대상에서 제외했다.

**의료기관 유형의 우선순위는 국립재활원 「2024년도 장애인 건강보건통계」의 장애인 요양기관별 의료이용 현황을 근거로 설정**하였다. ([국립재활원 자료실 – 2024년도 장애인 건강보건통계](https://www.nrc.go.kr/research/board/boardView.do?bn=newsView&board_id=NRC_NOTICE_BOARD&depart_no=0001&fno=23&menu_cd=05_02_00_02&no=24043&pageIndex=1)) 해당 통계는 장애인의 요양기관별 진료실인원과 내원일수를 제공하며, 상급종합병원·종합병원·병원·요양병원·정신병원·의원 등 기관 종별 의료이용을 구분하여 제시한다. 특히 장애유형별 분석에서도 의원, 병원, 종합병원, 상급종합병원 및 요양병원에서 상당한 의료이용이 확인되므로 이들을 장애인콜택시의 **핵심 의료 목적지 후보군(P1)**으로 선정하였다.

반면 정신병원, 보건소·보건지소 및 한의원은 장애인의 의료이용이 확인되지만 특정 장애유형 또는 특정 의료서비스에 대한 이용 성격이 상대적으로 강하므로 **지역사회·특수 의료 목적지 후보군(P2)**로 구분하였다.

이 근거를 아래 코드의 `P1_CODES`/`P2_CODES` 딕셔너리에 그대로 반영한다.

In [8]:
# 구 단위 결합에는 HIRA 응답의 sgguCdNm(시군구명)을 그대로 사용한다.
# 동 단위 결합에는 emdongNm(법정동) 대신 add_admin_dong.py로 추가한 '행정동' 컬럼을 사용한다(2-6절 참고).

CL_CD_NAME = {
    "01": "상급종합병원", "11": "종합병원", "21": "병원", "28": "요양병원", "31": "의원",
    "71": "보건소", "72": "보건지소", "93": "한의원", "29": "정신병원",
}
P1_CODES = {"01", "11", "21", "28", "31"}
P2_CODES = {"71", "72", "93", "29"}

df_hira["priority_group"] = df_hira["clCd"].map(
    lambda c: "P1" if c in P1_CODES else ("P2" if c in P2_CODES else "기타")
)
df_hira["clCd_name"] = df_hira["clCd"].map(CL_CD_NAME).fillna(df_hira["clCdNm"])

print(df_hira["priority_group"].value_counts())
print(f"\nsgguCdNm 결측(구 단위 집계 불가 대상): {df_hira['sgguCdNm'].isna().sum()}건")


priority_group
P1    11088
P2     3768
Name: count, dtype: int64

sgguCdNm 결측(구 단위 집계 불가 대상): 0건


### 2-6. 목적동(행정동) 표기 정규화 및 매칭 검증

콜택시 `목적동`(행정동)과 HIRA `행정동`(카카오 역지오코딩) 표기를 맞추기 위한 정규화 단계다. 시행착오를 포함해 시도한 방법과 근거를 그대로 기록한다.

| 시도 | 방법 | 매칭률(건수 기준) |
|---|---|---|
| ① | 원본 그대로 비교 (콜: 행정동 vs HIRA: 법정동 emdongNm) | 24.6% |
| ② | HIRA를 카카오 행정동으로 교체 (정규화 없음) | 58.0% |
| ③ | ② + '제' 글자 제거 (예: `신당제1동`→`신당1동`) | 96.8% |
| ④ | ③ + 행정동 개편 이력 매핑 (7개 구, 웹 검색으로 개편 시점·명칭 검증) | **99.83%** ← 채택 |

④에서 남는 0.17%(2,330건)는 추가 정규화로 해결되는 성격이 아니라 각각 별개 원인으로 확인되어 그대로 둔다:
- 용산구 서빙고동(1,723건): `data/raw/hospital_info_seoul.csv` 원본과 카카오 행정동 컬럼 양쪽에서 직접 확인한 결과, 해당 행정동에 P1+P2 의료기관이 **실제로 0개인 것을 확인함** (매칭 실패가 아니라 정상적인 0값)
- 중구 일부(영종동·동인천동 등, 607건): 인천광역시 중구 동명 혼입 의심 (구 이름 중복으로 인한 필터링 한계)

**행정동 개편 이력 매핑 근거** (모두 1:1 개칭 또는 N:1 통합이라 매핑에 모호함 없음, 웹 검색으로 검증):
- 노원구 `공릉1.3동` → `공릉1동` (2011.06.23 개칭)
- 강남구 `일원2동` → `개포3동` (2022.12.23 개칭)
- 강동구 `상일동` → `상일1동` (2021.07.01 개칭. 상일2동은 강일동에서 분리 신설된 별개 동이라 이 매핑과 무관)
- 중구 `신당제1~4동`, `신당제6동` → `신당동`/`다산동`/`약수동`/`청구동`/`동화동` (2013.07 개칭, 신당제5동은 그대로 유지)
- 동대문구 `답십리제3·4동`, `장안제3·4동`, `전농제3동`, `제기제1·2동` → `답십리1동`/`답십리2동`/`장안1동`/`장안2동`/`전농2동`/`제기동` (2009.05.04, 자치구 조례 제784호 — 22개 행정동을 14개로 통합)
- 동대문구 `이문제1·2동` → `이문1동`, `이문제3동` → `이문2동` (같은 조례 제784호. 구 이문1동+이문2동이 합쳐져 새 '이문1동'이 됐고, 구 이문3동이 '이문2동'으로 개칭됨 — **'이문2동'이라는 라벨이 개편 전후로 서로 다른 지역을 가리키는 사례**라 단순 '제' 제거만으로는 안 되고 반드시 이 매핑이 필요함. 출처: [동대문구청 - 행정동 연혁](https://www.ddm.go.kr/www/contents.do?key=259))
- 종로구 `명륜3가동` → `혜화동` (2012.08.01 폐지·흡수통합. 성균관대 주변을 관할하던 명륜3가동이 인근 혜화동에 흡수되어 현재는 혜화동 주민센터가 명륜1~4가 지역을 모두 관할함. 콜 데이터에는 신명칭 `혜화동`과 구명칭 `명륜3가동`이 공존하고 있었음)

⚠️ **발견한 데이터 정합성 문제**: 매핑 추가 전에는 `이문제2동`이 단순 '제' 제거만 거쳐 `이문2동`으로 변환됐는데, 이 문자열이 HIRA의 현재 `이문2동`(구 이문3동 지역)과 우연히 일치해 **잘못된 지역으로 매칭되고 있었다** (매칭은 됐지만 틀린 결과). 겉보기엔 매칭 성공이라 놓치기 쉬운 유형의 오류라, 아래 검증 셀에서 재배치가 제대로 됐는지 별도로 확인한다.

In [9]:
# ① 원본 그대로 비교: 콜(행정동) vs HIRA emdongNm(법정동)
hira_seoul = df_hira[df_hira["sgguCdNm"].isin(SEOUL_25)].copy()

call_dong_raw = completed_seoul_dest.dropna(subset=["목적동"])[["목적구", "목적동"]]
hira_bdong_set = set(zip(
    hira_seoul.dropna(subset=["emdongNm"])["sgguCdNm"],
    hira_seoul.dropna(subset=["emdongNm"])["emdongNm"],
))
matched_raw = sum(
    1 for gu, dong in zip(call_dong_raw["목적구"], call_dong_raw["목적동"])
    if (gu, dong) in hira_bdong_set
)
print(f"① 원본(법정동) 매칭률: {matched_raw:,}/{len(call_dong_raw):,} "
      f"({matched_raw / len(call_dong_raw) * 100:.1f}%)")


① 원본(법정동) 매칭률: 336,582/1,369,283 (24.6%)


In [10]:
import re

# ② 카카오 행정동으로 교체 (정규화 없음)
hira_hdong_set = set(zip(hira_seoul["sgguCdNm"], hira_seoul["행정동"]))
call_pairs_raw = list(zip(call_dong_raw["목적구"], call_dong_raw["목적동"]))
matched_hdong_raw = sum(1 for p in call_pairs_raw if p in hira_hdong_set)
print(f"② 카카오 행정동(정규화 전) 매칭률: {matched_hdong_raw:,}/{len(call_pairs_raw):,} "
      f"({matched_hdong_raw / len(call_pairs_raw) * 100:.1f}%)")


# ③ '제' 글자 제거: 콜택시 표기('신당제1동')를 카카오 표기('신당1동')에 맞춘다
def strip_je(name: str) -> str:
    return re.sub(r"제(\d+동)", r"\1", name)


call_dong_step3 = [strip_je(d) for d in call_dong_raw["목적동"]]
matched_step3 = sum(
    1 for gu, dong in zip(call_dong_raw["목적구"], call_dong_step3)
    if (gu, dong) in hira_hdong_set
)
print(f"③ ② + '제' 제거 매칭률: {matched_step3:,}/{len(call_dong_raw):,} "
      f"({matched_step3 / len(call_dong_raw) * 100:.1f}%)")


② 카카오 행정동(정규화 전) 매칭률: 794,764/1,369,283 (58.0%)


③ ② + '제' 제거 매칭률: 1,325,285/1,369,283 (96.8%)


In [11]:
# ④ ③ + 행정동 개편 이력 매핑 (웹 검색으로 검증된 값만 반영)
HISTORICAL_DONG_MAP = {
    ("노원구", "공릉1.3동"): "공릉1동",
    ("강남구", "일원2동"): "개포3동",
    ("강동구", "상일동"): "상일1동",
    ("중구", "신당제1동"): "신당동",
    ("중구", "신당제2동"): "다산동",
    ("중구", "신당제3동"): "약수동",
    ("중구", "신당제4동"): "청구동",
    ("중구", "신당제6동"): "동화동",
    ("동대문구", "답십리제3동"): "답십리1동",
    ("동대문구", "답십리제4동"): "답십리2동",
    ("동대문구", "제기제1동"): "제기동",
    ("동대문구", "제기제2동"): "제기동",
    ("동대문구", "장안제3동"): "장안1동",
    ("동대문구", "장안제4동"): "장안2동",
    ("동대문구", "전농제3동"): "전농2동",
    ("종로구", "명륜3가동"): "혜화동",
    ("동대문구", "이문제1동"): "이문1동",
    ("동대문구", "이문제2동"): "이문1동",  # 구 이문2동은 이문1동에 흡수 (단순 '제' 제거로는 이문2동으로 잘못 매칭됨)
    ("동대문구", "이문제3동"): "이문2동",  # 구 이문3동이 이문2동으로 개칭
}

before_rows = len(completed_seoul_dest)

completed_seoul_dest["목적동_정규화"] = [
    HISTORICAL_DONG_MAP.get((gu, dong), dong)
    for gu, dong in zip(completed_seoul_dest["목적구"], completed_seoul_dest["목적동"])
]
completed_seoul_dest["목적동_정규화"] = completed_seoul_dest["목적동_정규화"].apply(strip_je)

after_rows = len(completed_seoul_dest)
assert before_rows == after_rows, "정규화 과정에서 행 수가 변하면 카운팅 왜곡 의심"
print(f"[검증] 행 수 보존: {before_rows:,} == {after_rows:,}")

# [검증] 제기제1동+제기제2동(N:1 통합 사례) 합산이 정확히 보존되는지 확인
gijae_old = (
    (completed_seoul_dest["목적구"] == "동대문구")
    & completed_seoul_dest["목적동"].isin(["제기제1동", "제기제2동"])
).sum()
gijae_new = (
    (completed_seoul_dest["목적구"] == "동대문구")
    & (completed_seoul_dest["목적동_정규화"] == "제기동")
).sum()
print(f"[검증] 제기제1동+제기제2동 원본 합계: {gijae_old:,} / 정규화 후 제기동: {gijae_new:,} "
      f"(일치: {gijae_old == gijae_new})")

# [검증] 이문제2동이 (잘못된) 이문2동이 아니라 이문1동으로 재배치됐는지 확인
imun2_wrong = (
    (completed_seoul_dest["목적구"] == "동대문구")
    & (completed_seoul_dest["목적동"] == "이문제2동")
    & (completed_seoul_dest["목적동_정규화"] == "이문2동")
).sum()
imun2_correct = (
    (completed_seoul_dest["목적구"] == "동대문구")
    & (completed_seoul_dest["목적동"] == "이문제2동")
    & (completed_seoul_dest["목적동_정규화"] == "이문1동")
).sum()
print(f"[검증] 이문제2동 -> 잘못된 이문2동 매칭: {imun2_wrong:,}건 (0이어야 정상) / "
      f"올바른 이문1동 재배치: {imun2_correct:,}건")
assert imun2_wrong == 0, "이문제2동이 여전히 잘못된 이문2동으로 매칭되고 있음"

completed_seoul_dest["동_매칭됨"] = [
    (gu, dong) in hira_hdong_set
    for gu, dong in zip(completed_seoul_dest["목적구"], completed_seoul_dest["목적동_정규화"])
]
match_rate = completed_seoul_dest["동_매칭됨"].mean() * 100
print(f"\n④ 최종 매칭률: {completed_seoul_dest['동_매칭됨'].sum():,}/{len(completed_seoul_dest):,} "
      f"({match_rate:.2f}%)")

by_gu_match = (
    completed_seoul_dest
    .groupby("목적구")["동_매칭됨"]
    .agg(["sum", "count"])
    .assign(매칭률=lambda d: (d["sum"] / d["count"] * 100).round(2))
    .sort_values("매칭률")
)
display(by_gu_match)


[검증] 행 수 보존: 1,369,283 == 1,369,283


[검증] 제기제1동+제기제2동 원본 합계: 4,214 / 정규화 후 제기동: 4,214 (일치: True)


[검증] 이문제2동 -> 잘못된 이문2동 매칭: 0건 (0이어야 정상) / 올바른 이문1동 재배치: 698건



④ 최종 매칭률: 1,366,953/1,369,283 (99.83%)


,sum,count,매칭률
목적구,,,
용산구,24043,25766,93.31
중구,23457,24064,97.48
강남구,49717,49717,100.00
강동구,78684,78684,100.00
관악구,47767,47767,100.00
광진구,41518,41518,100.00
강북구,44792,44792,100.00
강서구,93425,93425,100.00
노원구,146831,146831,100.00


### 2-7. 데이터 결합 — 구 단위 (보조 산출물)

동 단위가 주 분석단위로 채택됐지만, 구 단위 롤업도 보조 산출물로 함께 만든다.

In [12]:
dest_gu_call_count = (
    completed_seoul_dest
    .groupby("목적구")
    .size()
    .reindex(SEOUL_25, fill_value=0)
    .rename("콜택시_목적지_건수")
    .reset_index()
    .rename(columns={"index": "목적구"})
)

display(dest_gu_call_count.sort_values("콜택시_목적지_건수", ascending=False).reset_index(drop=True))


,목적구,콜택시_목적지_건수
0,노원구,146831
1,강서구,93425
2,강동구,78684
3,서대문구,77346
4,은평구,74925
5,마포구,73304
6,영등포구,60518
7,중랑구,56697
8,송파구,54062
9,동대문구,53680


In [13]:
hira_seoul = df_hira[df_hira["sgguCdNm"].isin(SEOUL_25)].copy()

gu_cl_pivot = (
    hira_seoul
    .groupby(["sgguCdNm", "clCd_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(SEOUL_25, fill_value=0)
    .reset_index()
    .rename(columns={"sgguCdNm": "목적구"})
)

p1_cols = [CL_CD_NAME[c] for c in ["01", "11", "21", "28", "31"]]
p2_cols = [CL_CD_NAME[c] for c in ["71", "72", "93", "29"]]

gu_cl_pivot["P1_의료기관_합계"] = gu_cl_pivot[p1_cols].sum(axis=1)
gu_cl_pivot["P2_의료기관_합계"] = gu_cl_pivot[p2_cols].sum(axis=1)
gu_cl_pivot["의료기관_총수"] = gu_cl_pivot["P1_의료기관_합계"] + gu_cl_pivot["P2_의료기관_합계"]

display(gu_cl_pivot.sort_values("의료기관_총수", ascending=False).reset_index(drop=True))


clCd_name,목적구,병원,보건소,보건지소,상급종합병원,요양병원,의원,정신병원,종합병원,한의원,P1_의료기관_합계,P2_의료기관_합계,의료기관_총수
0,강남구,32,1,0,2,6,2178,0,2,397,2220,398,2618
1,서초구,14,1,2,1,4,987,0,1,271,1007,274,1281
2,송파구,22,1,1,1,7,690,1,1,273,721,276,997
3,강서구,20,1,0,0,4,489,1,4,205,517,207,724
4,강동구,18,1,0,0,5,483,0,3,197,509,198,707
5,영등포구,9,1,0,0,9,439,1,7,156,464,158,622
6,마포구,6,1,0,0,3,438,0,0,148,447,149,596
7,노원구,5,1,1,0,7,398,0,3,146,413,148,561
8,은평구,11,1,3,0,5,347,1,2,158,365,163,528
9,관악구,11,1,0,0,2,356,0,1,138,370,139,509


In [14]:
gu_combined = dest_gu_call_count.merge(gu_cl_pivot, on="목적구", how="left")

print(f"구 단위 결합 결과: {gu_combined.shape}")
display(gu_combined.sort_values("콜택시_목적지_건수", ascending=False).reset_index(drop=True))


구 단위 결합 결과: (25, 14)


,목적구,콜택시_목적지_건수,병원,보건소,보건지소,상급종합병원,요양병원,의원,정신병원,종합병원,한의원,P1_의료기관_합계,P2_의료기관_합계,의료기관_총수
0,노원구,146831,5,1,1,0,7,398,0,3,146,413,148,561
1,강서구,93425,20,1,0,0,4,489,1,4,205,517,207,724
2,강동구,78684,18,1,0,0,5,483,0,3,197,509,198,707
3,서대문구,77346,2,1,1,1,5,224,0,1,93,233,95,328
4,은평구,74925,11,1,3,0,5,347,1,2,158,365,163,528
5,마포구,73304,6,1,0,0,3,438,0,0,148,447,149,596
6,영등포구,60518,9,1,0,0,9,439,1,7,156,464,158,622
7,중랑구,56697,13,1,0,0,2,280,1,3,120,298,122,420
8,송파구,54062,22,1,1,1,7,690,1,1,273,721,276,997
9,동대문구,53680,12,1,0,1,5,299,3,3,149,320,153,473


### 2-8. 데이터 결합 — 구·동 단위 (주 분석단위)

2-6에서 검증된 정규화(`목적동_정규화`)를 기준으로 콜 목적지 건수와 HIRA 의료기관 종별 개수를 구·동 단위로 결합한다. 매칭 안 된 행정동은 outer 방식으로 살리고 의료기관 개수를 0으로 채운다 — 결측이 아니라 "그 동에 P1+P2 의료기관이 없다(또는 아직 매핑 안 된 잔여 0.28%)"는 뜻이므로 제외하지 않는다.

In [15]:
dest_dong_call_count = (
    completed_seoul_dest
    .groupby(["목적구", "목적동_정규화"])
    .size()
    .reset_index(name="콜택시_목적지_건수")
    .rename(columns={"목적동_정규화": "행정동"})
)

print(f"콜 목적(구,행정동) 조합 수: {len(dest_dong_call_count):,}")
display(dest_dong_call_count.sort_values("콜택시_목적지_건수", ascending=False).head(20))


콜 목적(구,행정동) 조합 수: 431


,목적구,행정동,콜택시_목적지_건수
141,노원구,상계6.7동,31372
218,서대문구,신촌동,27363
151,노원구,하계1동,25892
206,마포구,성산2동,20545
202,마포구,상암동,15420
384,종로구,이화동,15126
195,동작구,신대방2동,14564
341,영등포구,여의동,14308
427,중랑구,신내1동,13529
22,강동구,강일동,13505


In [16]:
dong_cl_pivot = (
    hira_seoul
    .groupby(["sgguCdNm", "행정동", "clCd_name"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"sgguCdNm": "목적구"})
)

for col in p1_cols + p2_cols:
    if col not in dong_cl_pivot.columns:
        dong_cl_pivot[col] = 0

dong_cl_pivot["P1_의료기관_합계"] = dong_cl_pivot[p1_cols].sum(axis=1)
dong_cl_pivot["P2_의료기관_합계"] = dong_cl_pivot[p2_cols].sum(axis=1)
dong_cl_pivot["의료기관_총수"] = dong_cl_pivot["P1_의료기관_합계"] + dong_cl_pivot["P2_의료기관_합계"]

print(f"HIRA (구,행정동) 조합 수: {len(dong_cl_pivot):,}")
display(dong_cl_pivot.sort_values("의료기관_총수", ascending=False).head(20))


HIRA (구,행정동) 조합 수: 428


clCd_name,목적구,행정동,병원,보건소,보건지소,상급종합병원,요양병원,의원,정신병원,종합병원,한의원,P1_의료기관_합계,P2_의료기관_합계,의료기관_총수
17,강남구,역삼1동,2,0,0,0,0,373,0,1,49,376,49,425
15,강남구,신사동,0,0,0,0,0,336,0,0,33,336,33,369
16,강남구,압구정동,4,0,0,0,2,255,0,0,42,261,42,303
4,강남구,논현1동,6,0,0,0,0,264,0,0,27,270,27,297
243,서초구,서초4동,1,0,0,0,0,245,0,0,40,246,40,286
21,강남구,청담동,4,0,0,0,1,245,0,0,29,250,29,279
5,강남구,논현2동,5,0,0,0,0,137,0,0,24,142,24,166
206,마포구,서교동,0,0,0,0,0,136,0,0,28,136,28,164
400,중구,명동,0,0,0,0,0,124,0,0,25,124,25,149
346,영등포구,여의동,0,0,0,0,0,117,0,1,30,118,30,148


In [17]:
dong_combined = dest_dong_call_count.merge(
    dong_cl_pivot, on=["목적구", "행정동"], how="left"
)

facility_cols = p1_cols + p2_cols + ["P1_의료기관_합계", "P2_의료기관_합계", "의료기관_총수"]
dong_combined[facility_cols] = dong_combined[facility_cols].fillna(0).astype(int)

zero_facility = dong_combined[dong_combined["의료기관_총수"] == 0]
print(f"구·동 결합 결과: {dong_combined.shape}")
print(f"의료기관 0건인 행정동: {len(zero_facility)}개 "
      f"(실제 미보유 지역 + 미해결 잔여 매칭실패 혼재 — 아래 목록 직접 확인)")
display(
    zero_facility[["목적구", "행정동", "콜택시_목적지_건수"]]
    .sort_values("콜택시_목적지_건수", ascending=False)
)

display(dong_combined.sort_values("콜택시_목적지_건수", ascending=False).head(20))


구·동 결합 결과: (431, 15)
의료기관 0건인 행정동: 8개 (실제 미보유 지역 + 미해결 잔여 매칭실패 혼재 — 아래 목록 직접 확인)


,목적구,행정동,콜택시_목적지_건수
346,용산구,서빙고동,1723
406,중구,영종동,511
407,중구,율목동,52
399,중구,북성동,20
403,중구,신흥동,17
395,중구,도원동,4
405,중구,연안동,2
396,중구,동인천동,1


,목적구,행정동,콜택시_목적지_건수,병원,보건소,보건지소,상급종합병원,요양병원,의원,정신병원,종합병원,한의원,P1_의료기관_합계,P2_의료기관_합계,의료기관_총수
141,노원구,상계6.7동,31372,1,1,0,0,1,109,0,1,34,112,35,147
218,서대문구,신촌동,27363,0,0,0,1,0,30,0,0,8,31,8,39
151,노원구,하계1동,25892,0,0,0,0,0,8,0,1,2,9,2,11
206,마포구,성산2동,20545,0,1,0,0,1,19,0,0,7,20,8,28
202,마포구,상암동,15420,0,0,0,0,1,31,0,0,12,32,12,44
384,종로구,이화동,15126,0,0,0,1,0,6,0,0,1,7,1,8
195,동작구,신대방2동,14564,0,0,0,0,0,17,0,1,4,18,4,22
341,영등포구,여의동,14308,0,0,0,0,0,117,0,1,30,118,30,148
427,중랑구,신내1동,13529,0,0,0,0,0,16,0,1,9,17,9,26
22,강동구,강일동,13505,0,0,0,0,0,20,0,0,7,20,7,27


### 2-9. 의료목적 필터링 병렬 데이터셋

지금까지 만든 `콜택시_목적지_건수`는 **이용목적과 무관하게 모든 도착 건수**다. 그런데 `이용목적` 컬럼을 확인해보면 '기타' 77.71%, '귀가' 7.70% 등 비의료 목적이 압도적이고, 치료/재활 계열로 명확히 표시된 건 약 9.2%(치료 5.92% + 재활 2.23% + 예약치료 0.68% + 예약재활 0.27% + 치료광역 0.10%)뿐이다. 연구 주제가 '의료 목적지 분석'인데 이 구분 없이 전체 목적지 건수를 그대로 쓰면, 실제로는 '동네 전체 생활 이동량 vs 의료기관 수'를 보는 셈이 되어 연구 질문과 어긋날 위험이 있다.

이를 해결하기 위해 **전체 목적지 건수(All)**와 **의료 목적지 건수(Medical, 이용목적 필터링)** 두 버전을 병렬로 만들어 07에서 나란히 비교한다. 어느 한쪽이 '정답'이라 단정하지 않는다 — Medical은 표본이 9%로 줄어드는 대신 연구 주제와 직접 연결되고, All은 표본은 크지만 해석이 간접적이다.

In [18]:
MEDICAL_PURPOSES = {"치료", "재활", "예약치료", "예약재활", "치료광역"}

purpose_counts = completed_seoul_dest["이용목적"].value_counts()
medical_ratio = completed_seoul_dest["이용목적"].isin(MEDICAL_PURPOSES).mean() * 100

print("이용목적 분포 (서울 25개구 목적지, 탑승완료 기준):")
display(purpose_counts)
print(f"\n의료 관련 이용목적({sorted(MEDICAL_PURPOSES)}) 비중: {medical_ratio:.2f}%")

completed_seoul_dest_medical = completed_seoul_dest[
    completed_seoul_dest["이용목적"].isin(MEDICAL_PURPOSES)
].copy()
print(f"\n의료 목적지 필터링 후 건수: {len(completed_seoul_dest_medical):,} / "
      f"{len(completed_seoul_dest):,} ({len(completed_seoul_dest_medical) / len(completed_seoul_dest) * 100:.2f}%)")

이용목적 분포 (서울 25개구 목적지, 탑승완료 기준):


이용목적
기타         1072326
귀가          103632
치료           82756
예약기타         42887
재활           29511
통학/출근        16024
예약치료         10192
종교            5290
예약재활          4189
예약통학/출근       1465
공항티켓           339
예약광역           189
쇼핑             160
예약귀가           146
심야예약           114
치료광역            57
업무               6
Name: count, dtype: int64


의료 관련 이용목적(['예약재활', '예약치료', '재활', '치료', '치료광역']) 비중: 9.25%

의료 목적지 필터링 후 건수: 126,705 / 1,369,283 (9.25%)


In [19]:
# 구 단위 (Medical)
dest_gu_call_count_medical = (
    completed_seoul_dest_medical
    .groupby("목적구")
    .size()
    .reindex(SEOUL_25, fill_value=0)
    .rename("콜택시_의료목적지_건수")
    .reset_index()
)

gu_combined_medical = dest_gu_call_count_medical.merge(gu_cl_pivot, on="목적구", how="left")

print(f"구 단위(Medical) 결합 결과: {gu_combined_medical.shape}")
display(gu_combined_medical.sort_values("콜택시_의료목적지_건수", ascending=False).reset_index(drop=True))

구 단위(Medical) 결합 결과: (25, 14)


,목적구,콜택시_의료목적지_건수,병원,보건소,보건지소,상급종합병원,요양병원,의원,정신병원,종합병원,한의원,P1_의료기관_합계,P2_의료기관_합계,의료기관_총수
0,노원구,11531,5,1,1,0,7,398,0,3,146,413,148,561
1,강동구,9692,18,1,0,0,5,483,0,3,197,509,198,707
2,서대문구,9587,2,1,1,1,5,224,0,1,93,233,95,328
3,강서구,7295,20,1,0,0,4,489,1,4,205,517,207,724
4,영등포구,7292,9,1,0,0,9,439,1,7,156,464,158,622
5,동대문구,6664,12,1,0,1,5,299,3,3,149,320,153,473
6,동작구,5994,5,1,0,1,1,298,0,1,133,306,134,440
7,은평구,5902,11,1,3,0,5,347,1,2,158,365,163,528
8,강남구,5287,32,1,0,2,6,2178,0,2,397,2220,398,2618
9,마포구,5248,6,1,0,0,3,438,0,0,148,447,149,596


In [20]:
# 구·동 단위 (Medical) — 2-6에서 만든 목적동_정규화를 그대로 재사용
dest_dong_call_count_medical = (
    completed_seoul_dest_medical
    .groupby(["목적구", "목적동_정규화"])
    .size()
    .reset_index(name="콜택시_의료목적지_건수")
    .rename(columns={"목적동_정규화": "행정동"})
)

dong_combined_medical = dest_dong_call_count_medical.merge(
    dong_cl_pivot, on=["목적구", "행정동"], how="left"
)
dong_combined_medical[facility_cols] = dong_combined_medical[facility_cols].fillna(0).astype(int)

print(f"구·동 단위(Medical) 결합 결과: {dong_combined_medical.shape}")
display(dong_combined_medical.sort_values("콜택시_의료목적지_건수", ascending=False).head(20))

구·동 단위(Medical) 결합 결과: (416, 15)


,목적구,행정동,콜택시_의료목적지_건수,병원,보건소,보건지소,상급종합병원,요양병원,의원,정신병원,종합병원,한의원,P1_의료기관_합계,P2_의료기관_합계,의료기관_총수
214,서대문구,신촌동,6420,0,0,0,1,0,30,0,0,8,31,8,39
138,노원구,상계6.7동,5136,1,1,0,0,1,109,0,1,34,112,35,147
192,동작구,신대방2동,3975,0,0,0,0,0,17,0,1,4,18,4,22
26,강동구,둔촌2동,3169,0,0,0,0,1,9,0,1,7,11,7,18
375,종로구,이화동,3156,0,0,0,1,0,6,0,0,1,7,1,8
148,노원구,하계1동,2930,0,0,0,0,0,8,0,1,2,9,2,11
301,송파구,풍납2동,2111,0,0,0,1,0,10,0,0,6,11,6,17
412,중랑구,신내1동,2061,0,0,0,0,0,16,0,1,9,17,9,26
59,강서구,방화1동,1972,2,0,0,0,0,46,0,0,18,48,18,66
114,구로구,구로2동,1769,1,0,0,1,0,34,0,0,5,36,5,41


### 중간 산출물 저장

| 파일 | 설명 | 컬럼 |
|---|---|---|
| `region_dong_call_hospital_combined.csv` | 구·동 단위, **전체 목적지**(All) 기준 (주 분석단위) | 목적구, 행정동, 콜택시_목적지_건수, 종별 9종 개수, P1/P2/총 의료기관수 |
| `region_gu_call_hospital_combined.csv` | 구 단위, 전체 목적지(All) 기준 (보조) | 목적구, 콜택시_목적지_건수, 종별 9종 개수, P1/P2/총 의료기관수 |
| `region_dong_call_hospital_combined_medical.csv` | 구·동 단위, **의료 목적지만**(Medical, 이용목적 필터링) 기준 | 목적구, 행정동, 콜택시_의료목적지_건수, 종별 9종 개수, P1/P2/총 의료기관수 |
| `region_gu_call_hospital_combined_medical.csv` | 구 단위, 의료 목적지만(Medical) 기준 | 목적구, 콜택시_의료목적지_건수, 종별 9종 개수, P1/P2/총 의료기관수 |

`이용건 정의 = 탑승완료(승차·하차 O, 취소 X)`, `목적구 = 서울 25개 자치구만 포함`, `행정동 = 2-6절 정규화 로직 적용(매칭률 99.83%)`, `의료기관 = data/processed/hospital_info_seoul_행정동_파생컬럼추가.csv 전체(P1+P2)`, `Medical 필터 = 이용목적 ∈ {치료, 재활, 예약치료, 예약재활, 치료광역}` 기준으로 생성됨. 07에서 All과 Medical 두 버전을 나란히 비교한다.

In [21]:
dong_output_path = OUTPUT_DIR / "region_dong_call_hospital_combined.csv"
dong_combined.to_csv(dong_output_path, index=False, encoding="utf-8-sig")
print(f"saved: {dong_output_path}")
print(f"rows: {len(dong_combined)}, cols: {len(dong_combined.columns)}")

gu_output_path = OUTPUT_DIR / "region_gu_call_hospital_combined.csv"
gu_combined.to_csv(gu_output_path, index=False, encoding="utf-8-sig")
print(f"\nsaved: {gu_output_path}")
print(f"rows: {len(gu_combined)}, cols: {len(gu_combined.columns)}")

dong_medical_output_path = OUTPUT_DIR / "region_dong_call_hospital_combined_medical.csv"
dong_combined_medical.to_csv(dong_medical_output_path, index=False, encoding="utf-8-sig")
print(f"\nsaved: {dong_medical_output_path}")
print(f"rows: {len(dong_combined_medical)}, cols: {len(dong_combined_medical.columns)}")

gu_medical_output_path = OUTPUT_DIR / "region_gu_call_hospital_combined_medical.csv"
gu_combined_medical.to_csv(gu_medical_output_path, index=False, encoding="utf-8-sig")
print(f"\nsaved: {gu_medical_output_path}")
print(f"rows: {len(gu_combined_medical)}, cols: {len(gu_combined_medical.columns)}")


saved: C:\Users\young\my_project\260901_call_taxi_DA\data\processed\region_dong_call_hospital_combined.csv
rows: 431, cols: 15

saved: C:\Users\young\my_project\260901_call_taxi_DA\data\processed\region_gu_call_hospital_combined.csv
rows: 25, cols: 14

saved: C:\Users\young\my_project\260901_call_taxi_DA\data\processed\region_dong_call_hospital_combined_medical.csv
rows: 416, cols: 15

saved: C:\Users\young\my_project\260901_call_taxi_DA\data\processed\region_gu_call_hospital_combined_medical.csv
rows: 25, cols: 14
